# PIE paper-weights reproduction on the full test split (Colab)

Runs the published Keras `.h5` checkpoints through our PyTorch port on **set03 (the paper's test split)** and reports intent / trajectory / speed metrics against the paper's numbers.

## Reference numbers from the paper / committed checkpoint filenames

| Model | Metric | Paper value |
|---|---|---|
| Intent (crossing) | accuracy | ≈ 0.81 (`intention/context_loc_pretrained/0.81.pkl`) |
| Trajectory | MSE at 45 steps | ≈ 473 (`trajectory/loc_intent_speed_pretrained/473.14.pkl`) |
| Speed | MSE at 45 steps | ≈ 2.65 (`speed/speed_pretrained/2.65.pkl`) |

## Resource budget

Reproducing on set03 requires ~25 GB of MP4s + ~15-25 GB of extracted frames + ~8 GB of VGG16 feature cache on Drive. **Free Colab Drive (15 GB) is insufficient.** Colab-Pro (100+ GB) or a local run is needed. The preconditions cell below checks this and fails loudly if there isn't enough space.

Wall-clock on a T4:
- set03 videos download: ~5-15 min depending on bandwidth
- set03 frame extraction: ~15-30 min
- Intent VGG feature caching: ~30-60 min (one-shot)
- Evals after caches are warm: minutes each

Budget a **full session (2-3 hours)** on first run; subsequent runs reuse the caches and complete in ~10 minutes.

## 1. Runtime + Drive

In [ ]:
# GPU is required for a reasonable wall-clock; CPU will take many hours.
!nvidia-smi | head -5 || echo 'No GPU detected - Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ['PIE_PATH'] = '/content/drive/MyDrive/Portofolios/Protofolio_2026/pedestrian_estimation_pie/pie_data'
print('PIE_PATH =', os.environ['PIE_PATH'])

## 2. Clone + install

In [ ]:
%cd /content
![ -d PIE_Pedestrian_Estimation ] || git clone -b claude/tensorflow-to-pytorch-conversion-0SlwI https://github.com/venetisgr/PIE_Pedestrian_Estimation.git
%cd /content/PIE_Pedestrian_Estimation
!git checkout claude/tensorflow-to-pytorch-conversion-0SlwI
!git pull
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

## 3. Precondition check

This cell will **raise** (red-bordered error) if any of these are missing or unusable. Don't skip the resulting guidance — downstream cells will fail opaquely without it.

In [ ]:
import os, shutil
from pathlib import Path

PIE_PATH = Path(os.environ['PIE_PATH'])
assert PIE_PATH.exists(), f'PIE_PATH does not exist: {PIE_PATH}. Create it on Drive or edit the path above.'

# Drive space check. set03 needs ~50 GB for a full pass.
free_gb = shutil.disk_usage(str(PIE_PATH)).free / 2**30
print(f'Drive free space: {free_gb:.1f} GB')
if free_gb < 35:
    print(f'WARNING: only {free_gb:.1f} GB free. set03 full pass needs ~35+ GB. '
          'You can still proceed if videos + frames are already extracted; otherwise '
          'use Colab Pro or free up space.')

# Verify the paper .h5 checkpoints are committed in the repo.
H5 = {
    'intent':     Path('data/pie/intention/context_loc_pretrained/model.h5'),
    'trajectory': Path('data/pie/trajectory/loc_intent_speed_pretrained/model.h5'),
    'speed':      Path('data/pie/speed/speed_pretrained/model.h5'),
}
for task, p in H5.items():
    assert p.is_file(), f'Paper checkpoint for {task} missing: {p}. '  \
        'This file should be in the repo under data/pie/...; check git status.'
print('paper .h5 checkpoints present:', {k: str(v) for k, v in H5.items()})

In [ ]:
# Annotations check + auto-fetch if missing.
required_annot_dirs = ['annotations', 'annotations_attributes', 'annotations_vehicle']
missing_annot = [d for d in required_annot_dirs if not (PIE_PATH / d / 'set03').is_dir()]
if missing_annot:
    print('Annotations missing for', missing_annot, '- downloading...')
    !python -m pie_pytorch.data.downloader annotations --dest "$PIE_PATH" --overwrite
else:
    print('Annotations present: ok')

In [ ]:
# set03 videos check + auto-fetch if missing. 19 files, ~25 GB.
import glob

set03_dir = PIE_PATH / 'PIE_clips' / 'set03'
set03_videos = sorted(glob.glob(str(set03_dir / 'video_*.mp4')))
print(f'set03 videos currently present: {len(set03_videos)}/19')
if len(set03_videos) < 19:
    print('Downloading missing set03 videos (resumable; ~25 GB total)...')
    !python -m pie_pytorch.data.downloader videos --dest "$PIE_PATH" --sets set03 --workers 4
else:
    print('set03 videos present: ok')

In [ ]:
# set03 annotated-frame extraction check + auto-run if missing.
# We check by looking at one video dir; full extraction takes ~15-30 min.
sample_video_frames = PIE_PATH / 'images' / 'set03' / 'video_0001'
needs_extract = not sample_video_frames.is_dir() or not list(sample_video_frames.glob('*.png'))

if needs_extract:
    print('Extracting annotated frames for all videos on disk (one-shot, ~15-30 min for set03)...')
    from pie_pytorch.data.pie_data import PIE
    imdb = PIE(data_path=str(PIE_PATH))
    imdb.extract_and_save_images(extract_frame_type='annotated')
    print('frame extraction done.')
else:
    print('set03 frames already extracted: ok')

In [ ]:
# Final green-light check before evals.
from pie_pytorch.data.pie_data import PIE

imdb = PIE(data_path=str(PIE_PATH))
ann = imdb.get_annotated_frame_numbers('set03')
n_vids = len(ann)
n_frames = sum(len(v) for v in ann.values())
print(f'set03 annotated: {n_vids} videos, {n_frames} total frames')
assert n_vids == 19, f'set03 should have 19 videos of annotations, got {n_vids}'
print('preconditions OK - ready to evaluate')

## 4. Convert the three `.h5` checkpoints (one-shot)

In [ ]:
!python -m pie_pytorch.cli.convert \
    --task intent \
    --h5 data/pie/intention/context_loc_pretrained/model.h5 \
    --out data/pie/intention/context_loc_pretrained/model.safetensors
!python -m pie_pytorch.cli.convert \
    --task trajectory \
    --h5 data/pie/trajectory/loc_intent_speed_pretrained/model.h5 \
    --out data/pie/trajectory/loc_intent_speed_pretrained/model.safetensors
!python -m pie_pytorch.cli.convert \
    --task speed \
    --h5 data/pie/speed/speed_pretrained/model.h5 \
    --out data/pie/speed/speed_pretrained/model.safetensors

## 5. Evaluate paper weights on **set03** (paper's test split)

We override the YAML's `val_split` → `test` and drop the set05 subset filter so the eval runs on the full test set (set03, 19 videos).

### 5a. Intent (paper: accuracy ≈ 0.81)

First run caches VGG16 features per window — slow (~30–60 min). Subsequent runs hit the cache.

In [ ]:
!python -m pie_pytorch.cli.eval \
    --config pie_pytorch/configs/intent_colab.yaml \
    --keras-h5 data/pie/intention/context_loc_pretrained/model.h5 \
    --split test

### 5b. Trajectory (paper: MSE ≈ 473)

In [ ]:
!python -m pie_pytorch.cli.eval \
    --config pie_pytorch/configs/trajectory_colab.yaml \
    --keras-h5 data/pie/trajectory/loc_intent_speed_pretrained/model.h5 \
    --split test

### 5c. Speed (paper: MSE ≈ 2.65)

The paper trained speed with a 1-dim zero-filled decoder input; our `speed_colab.yaml` uses `dec_feature_size: 0` for scratch-training. `pie-eval --keras-h5` autosizes the model from the `.h5` shapes (which encode `dec_feature_size=1`), so the eval is correct regardless.

In [ ]:
!python -m pie_pytorch.cli.eval \
    --config pie_pytorch/configs/speed_colab.yaml \
    --keras-h5 data/pie/speed/speed_pretrained/model.h5 \
    --split test

## Done

Paste the three metric blocks back to the thread. Expected:
- Intent acc ≈ 0.79–0.82, F1 ≈ 0.85–0.90.
- Trajectory MSE ≈ 400–500.
- Speed MSE ≈ 2–4.

Anything drastically off → paste the full eval output + the first 5 lines of `!du -sh $PIE_PATH/*`, and I'll diagnose.